# CaFE-Rec: Training on Google Colab (T4 GPU)

This notebook lets you train the CaFE-Rec model on a free Colab T4 GPU.

**Before running this notebook:**
1. Upload the `cafe_rec/` folder to your Google Drive
2. Upload the `cafe_rec/data/processed/` folder (train.pkl, val.pkl, test.pkl, aspect_vocab.json, mappings.json)
3. Set runtime to **GPU** (Runtime -> Change runtime type -> T4 GPU)

## Step 1: Check GPU is available

In [ ]:
import torch, os, multiprocessing as mp

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    n_gpu = torch.cuda.device_count()
    print(f"GPU(s): {n_gpu} x {name}  ({mem_gb:.1f} GB each)")

    # Recommend flags by GPU class.
    if "A100" in name or "H100" in name:
        rec = dict(batch_size=256, max_length=96, num_workers=8, amp=True)
    elif "L4" in name or "V100" in name:
        rec = dict(batch_size=128, max_length=80, num_workers=8, amp=True)
    elif "T4" in name or "P100" in name:
        rec = dict(batch_size=64, max_length=64, num_workers=4, amp=True)
    else:
        rec = dict(batch_size=32, max_length=64, num_workers=4, amp=True)

    # Cap workers at the actual CPU count.
    rec["num_workers"] = min(rec["num_workers"], mp.cpu_count())
    print(f"\nRecommended flags for this GPU: {rec}")
    # Stash for later cells.
    REC_FLAGS = rec
    N_GPU = n_gpu
else:
    print("WARNING: No GPU found! Go to Runtime -> Change runtime type -> T4 GPU")
    REC_FLAGS = dict(batch_size=16, max_length=64, num_workers=2, amp=False)
    N_GPU = 0


## Step 2: Mount Google Drive

Your project folder should be in Google Drive. Update the path below if needed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# UPDATE THIS PATH to match where you put cafe_rec/ in Google Drive
PROJECT_DIR = "/content/drive/MyDrive/Recommender_Project/cafe_rec"

import os
os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")
print(f"Files: {os.listdir('.')}")

## Step 3: Install dependencies

In [ ]:
!pip install -q torch transformers torch-geometric datasets wandb \
    numpy pandas scikit-learn tqdm sacrebleu rouge-score bert-score \
    sentencepiece accelerate spacy
!python -m spacy download en_core_web_sm -q
print("All dependencies installed!")

## Step 4: Verify data is in place

In [ ]:
required_files = [
    "data/processed/train.pkl",
    "data/processed/val.pkl",
    "data/processed/test.pkl",
    "data/processed/aspect_vocab.json",
    "data/processed/mappings.json",
]

all_ok = True
for f in required_files:
    exists = os.path.exists(f)
    size = f"{os.path.getsize(f) / 1e6:.1f} MB" if exists else "MISSING"
    status = "OK" if exists else "MISSING"
    print(f"  {status:7s} {size:>10s}  {f}")
    if not exists:
        all_ok = False

if all_ok:
    print("\nAll data files found! Ready to train.")
else:
    print("\nSome files are missing. Upload them to Google Drive first.")

### Optional: clean up old broken checkpoints

Previous broken runs (NaN losses, bogus Recall=1.0) left ~24 GB of useless `.pt` files in `checkpoints/`. The cell below removes them so the new run starts from a clean slate. Only run if you don't need any old checkpoints.


In [ ]:
# Wipe old checkpoints (frees ~24 GB on Drive). Skip if you want to keep them.
import glob, os
old = glob.glob('checkpoints/*.pt')
for p in old:
    sz = os.path.getsize(p) / 1e9
    os.remove(p)
    print(f'removed {sz:.2f} GB  {p}')
print(f'done -- removed {len(old)} files')


## Step 4b: One-time aspect-label precompute (saves ~40 min per future run)

`train.py` re-encodes aspect labels for all 6M+ rows on every fresh Colab session unless they're already baked into the pickles. Run this cell **once** -- it rewrites `train.pkl` / `val.pkl` / `test.pkl` in place to include the `aspect_labels` column. After that, every future training session skips the 40-min encoding step entirely.

If you re-run this cell later it will detect the column is already present and skip immediately, so it's safe to leave in.


In [ ]:
# One-time: bake aspect_labels into the .pkl files in data/processed/.
# Takes ~40 min on first run; instant on subsequent runs.
!python -m data.precompute_aspects --processed_dir data/processed


## Step 5: Quick smoke test (optional)

Run this to verify everything works before starting the long training.

In [ ]:
!python train.py --dataset amazon_movies --processed_dir data/processed --no_wandb --debug

## Step 6: Train the model

Four training presets. Pick one by editing `RUN_OPTION` in the next cell.

| | What it does | Wall-clock (L4) | Compute units | Use when |
|---|---|---|---|---|
| **A. Fast** | Flan-T5-small + 100k sample + AMP | ~25 min | ~2 | Quick iteration / sanity check |
| **B. Quality** | Flan-T5-base + 500k sample + 5 epochs + AMP | ~6-7 hr | ~30-35 | Solid baseline run |
| **C. Multi-GPU** | DDP via `torchrun`, N_GPU >= 2 | Near-linear in GPU count | varies | Pro+ or external compute only |
| **D. Paper-grade** | Flan-T5-base + 500k sample + 8 epochs + AMP, per-epoch Drive backup | ~8 hr | ~40 | Final paper checkpoint with faithfulness training |

**Why not PySpark?** Spark parallelizes *data jobs*, not deep-model training. The right tool for distributing training is `torch.distributed` / `torchrun` (Option C). Use Spark only for offline aspect extraction.


In [ ]:
# Set which training option to run, and write fast_config.json (used by A & C).
# Options: "A" (fast/100k), "B" (quality/500k), "C" (multi-GPU), "D" (full data, paper-grade).
import json

RUN_OPTION = "A"   # <-- change to "B", "C", or "D"

fast_overrides = {
    "flan_model_name": "google/flan-t5-small",
    "adapter_dim": 512,
    "embedding_dim": 64,
    "warmup_epochs": 1,
}
with open("fast_config.json", "w") as f:
    json.dump(fast_overrides, f, indent=2)
print(f"RUN_OPTION = {RUN_OPTION}")
print("Wrote fast_config.json:")
print(json.dumps(fast_overrides, indent=2))


In [ ]:
# OPTION A: FAST -- Flan-T5-small + AMP + tuned batch (recommended first run)
if RUN_OPTION != "A":
    print(f"[skip] RUN_OPTION is {RUN_OPTION!r}, not 'A' -- skipping Option A.")
else:
    flags = REC_FLAGS
    cmd = f'''python train.py \\
        --dataset amazon_movies \\
        --processed_dir data/processed \\
        --config fast_config.json \\
        --no_wandb \\
        --sample_size 100000 \\
        --batch_size {flags["batch_size"]} \\
        --max_length {flags["max_length"]} \\
        --num_workers {flags["num_workers"]} \\
        {"--amp" if flags["amp"] else ""}'''
    print("Running:\n", cmd, "\n")
    get_ipython().system(cmd)


In [ ]:
# OPTION B: QUALITY -- Flan-T5-base with tuned flags (no config override)
# Use this for the paper-grade run. Slower than A but matches the original architecture.
if RUN_OPTION != "B":
    print(f"[skip] RUN_OPTION is {RUN_OPTION!r}, not 'B' -- skipping Option B.")
else:
    flags = REC_FLAGS
    bs = max(flags["batch_size"] // 2, 16)   # halve batch for the larger model
    cmd = f'''python train.py \\
        --dataset amazon_movies \\
        --processed_dir data/processed \\
        --no_wandb \\
        --sample_size 500000 \\
        --batch_size {bs} \\
        --max_length {flags["max_length"]} \\
        --num_workers {flags["num_workers"]} \\
        {"--amp" if flags["amp"] else ""}'''
    print("Running:\n", cmd, "\n")
    get_ipython().system(cmd)


In [ ]:
# OPTION C: MULTI-GPU via torchrun (only useful if N_GPU >= 2).
if RUN_OPTION != "C":
    print(f"[skip] RUN_OPTION is {RUN_OPTION!r}, not 'C' -- skipping Option C.")
elif N_GPU < 2:
    print(f"[skip] Only {N_GPU} GPU(s) detected -- Option C needs >= 2.")
else:
    flags = REC_FLAGS
    per_gpu_bs = flags["batch_size"]
    cmd = f'''torchrun --standalone --nproc_per_node={N_GPU} train.py \\
        --dataset amazon_movies \\
        --processed_dir data/processed \\
        --config fast_config.json \\
        --no_wandb \\
        --batch_size {per_gpu_bs} \\
        --max_length {flags["max_length"]} \\
        --num_workers {flags["num_workers"]} \\
        --amp'''
    print(f"Launching DDP across {N_GPU} GPUs (effective batch = {per_gpu_bs * N_GPU}):\n", cmd, "\n")
    get_ipython().system(cmd)


In [ ]:
# OPTION D: PAPER-GRADE -- 500k sample + 8 epochs + Flan-T5-base + AMP.
# Designed to fit in ~8 hr on L4 (well within Colab Pro's 12-hr session limit).
# Writes checkpoints to /content/ (fast atomic local SSD) and streams each new 'best'
# checkpoint to Drive via a background thread -- if Colab disconnects, the latest best
# is already safely on Drive.
SEED = 0
SAMPLE_SIZE = 500_000     # 500k training rows (vs Option B's 500k, but with 60% more epochs)
NUM_EPOCHS  = 8           # post-warmup epochs do the faithfulness training
if RUN_OPTION != "D":
    print(f"[skip] RUN_OPTION is {RUN_OPTION!r}, not 'D' -- skipping Option D.")
else:
    import os, shutil, glob, threading, time
    flags = REC_FLAGS
    bs = max(flags["batch_size"] // 2, 16)   # halve batch for the larger model
    local_save = f"/content/checkpoints/seed{SEED}"
    drive_save = f"checkpoints/seed{SEED}"
    os.makedirs(local_save, exist_ok=True)
    os.makedirs(drive_save, exist_ok=True)

    # --- Background backup thread: copies each new best_*.pt from /content/ to Drive,
    # verifies size, and uses tmp-then-rename to avoid Drive FUSE half-writes.
    stop_evt = threading.Event()
    copied = set()

    def _safe_copy(src_path: str) -> bool:
        size = os.path.getsize(src_path)
        time.sleep(3)
        if os.path.getsize(src_path) != size:
            return False  # still being written
        name = os.path.basename(src_path)
        tmp = os.path.join(drive_save, name + ".tmp")
        final = os.path.join(drive_save, name)
        print(f"\n[backup] copying {name} ({size/1e9:.2f} GB) -> Drive ...", flush=True)
        try:
            shutil.copy2(src_path, tmp)
            drive_sz = os.path.getsize(tmp)
            if drive_sz != size:
                os.remove(tmp)
                print(f"[backup] SIZE MISMATCH ({drive_sz} vs {size}); will retry.")
                return False
            os.replace(tmp, final)
            print(f"[backup] OK -> {final}")
            return True
        except Exception as e:
            print(f"[backup] error copying {name}: {e}")
            if os.path.exists(tmp):
                try: os.remove(tmp)
                except: pass
            return False

    def _backup_loop():
        while not stop_evt.is_set():
            for p in glob.glob(f"{local_save}/*_best_*.pt"):
                if p in copied:
                    continue
                if _safe_copy(p):
                    copied.add(p)
            stop_evt.wait(60)

    backup_thread = threading.Thread(target=_backup_loop, daemon=True)
    backup_thread.start()
    print(f"[backup] watching {local_save}/*_best_*.pt -- copies to {drive_save}/ every 60 sec")

    cmd = f'''python train.py \\
        --dataset amazon_movies \\
        --processed_dir data/processed \\
        --no_wandb \\
        --sample_size {SAMPLE_SIZE} \\
        --num_epochs {NUM_EPOCHS} \\
        --batch_size {bs} \\
        --max_length {flags["max_length"]} \\
        --num_workers {flags["num_workers"]} \\
        --seed {SEED} \\
        --save_dir {local_save} \\
        {"--amp" if flags["amp"] else ""}'''
    print(f"Paper-grade run: seed={SEED}, sample={SAMPLE_SIZE:,}, epochs={NUM_EPOCHS}, model=Flan-T5-base")
    print("Running:\n", cmd, "\n")
    try:
        get_ipython().system(cmd)
    finally:
        print("\n[backup] training finished; doing final backup pass...")
        stop_evt.set()
        backup_thread.join(timeout=5)
        for p in glob.glob(f"{local_save}/*_best_*.pt"):
            if p not in copied:
                _safe_copy(p)
        on_drive = sorted(glob.glob(f"{drive_save}/*_best_*.pt"), key=os.path.getmtime)
        print(f"\n[backup] {len(on_drive)} best-checkpoint(s) on Drive:")
        for p in on_drive:
            print(f"  {os.path.getsize(p)/1e9:.2f} GB  {p}")


### Parallel-seed runs (paper variance estimates)

Colab Pro caps you at **1 GPU session per account at a time**. To run multiple seeds in parallel you need either:
1. **Colab Pro+** (up to 3 simultaneous GPU sessions), OR
2. **Multiple Google accounts**, each running one seed in its own browser/Colab tab.

If you have either, launch each seed in its own notebook session with `SEED = 0`, `SEED = 1`, `SEED = 2` in Option D. Because Option D writes to `checkpoints/seed{SEED}/`, the runs won't clobber each other on Drive. Average the final test metrics across seeds and report mean +/- std in the paper.

If you only have Pro (single session): run seeds **sequentially** — set `SEED = 0`, run Option D, wait ~6 hr, then `SEED = 1`, re-run, etc. With 277 compute units and ~30 units per full-data run, you have headroom for **~8 sequential full-data runs**.


## Step 6b: Validate the checkpoint (30 seconds, do this before Option D)

After Option A finishes, run this to check whether the model actually learned anything -- or whether it silently failed like the previous run did. Six checks:

1. **weights finite** -- no NaN/inf anywhere in the model.
2. **scores non-degenerate** -- different (u, i) pairs get different scores (not all-equal).
3. **aspect selector sparse** -- picks ~K aspects per row, not 0 or all 50.
4. **fidelity drop** -- removing selected aspects decreases the score (faithful explanation).
5. **recall realistic** -- Recall@10 on a 100-row sample is in `[0.02, 0.99]` (not the bogus 1.0).
6. **explanation gen works** -- Flan-T5 produces a non-empty text string.

Only commit to Option D if all six pass.


In [ ]:
# Auto-pick the most recent 'best' checkpoint and validate it.
import glob, os
ckpts = sorted(
    [c for c in glob.glob('checkpoints/**/*.pt', recursive=True) if 'best' in c],
    key=os.path.getmtime,
)
if not ckpts:
    print('No best-checkpoint .pt files found in checkpoints/. Run an option first.')
else:
    latest = ckpts[-1]
    print(f'Validating: {latest}\n')
    get_ipython().system(f'python validate_run.py --checkpoint "{latest}" --processed_dir data/processed')


## Step 7: Evaluate the trained model

After training finishes, check the results. The best checkpoint is auto-saved.

In [ ]:
# List saved checkpoints
import glob
ckpts = sorted(glob.glob("checkpoints/*.pt"))
for c in ckpts:
    size = os.path.getsize(c) / 1e6
    print(f"  {size:>8.1f} MB  {c}")

if ckpts:
    print(f"\nBest checkpoint: {ckpts[-1]}")

In [ ]:
# Run evaluation on the best checkpoint
!python evaluate.py --help

## Step 8: Download results

Checkpoints are saved in `checkpoints/` on your Google Drive, so they persist after Colab disconnects.

In [ ]:
# Copy checkpoints to a safe location on Drive
!cp -r checkpoints/ /content/drive/MyDrive/Recommender_Project/checkpoints_backup/
print("Checkpoints backed up to Google Drive!")